# Module 08: Object Serialization with Pickle
## Notebook 02: Custom State Hooks, Class Versioning, and `__reduce__`

Default pickling serializes an object's `__dict__`. However, real-world machine learning systems contain state that **cannot or should not be pickled** (e.g., active database connections, open log file handles, threading locks, temporary GPU memory buffers).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand how Python reconstructs instances without calling `__init__()`.
2. Intercept serialization using the **`__getstate__()`** and **`__setstate__()`** hooks.
3. Exclude unpicklable or transient resources (sockets, locks, file pointers).
4. Handle **Class Schema Evolution & Versioning** to maintain backwards compatibility across software updates.
5. **Advanced:** Master the **`__reduce__()` protocol** for low-level control over object instantiation and factory reconstruction.
6. **Advanced:** Construct a production-grade **Stateful Feature Transformer Class** that safely decouples training scratchpads from serialized inference artifacts.

In [ ]:
import os
import sys
import pickle
import time
import numpy as np

print(f"Pickle module loaded. Python version: {sys.version.split()[0]}")

### 1. The `__init__` Bypassing Pitfall
A critical realization in Python serialization: **`pickle.load()` DOES NOT invoke `__init__()`!**
- Instead, pickle creates an empty uninitialized instance via `object.__new__(cls)` and directly populates its `__dict__` with the serialized dictionary.
- If your class relies on `__init__` to establish default connections, initialize caches, or create runtime locks, those fields will be missing unless managed via `__setstate__`!

In [ ]:
class ModelLogger:
    def __init__(self, log_name):
        self.log_name = log_name
        self.created_at = time.time()
        # Open file handle (UNPICKLABLE!)
        self.file_handle = open(f"/tmp/{log_name}.log", "w")
        self.file_handle.write(f"Logger initialized at {self.created_at}\n")

    def log(self, message):
        self.file_handle.write(f"[{time.strftime('%H:%M:%S')}] {message}\n")
        self.file_handle.flush()

logger = ModelLogger("experiment_alpha")
logger.log("Training started...")

# Attempting standard pickle fails because file handles cannot be serialized
try:
    pickle.dumps(logger)
except (TypeError, pickle.PicklingError, AttributeError) as e:
    print(f"Caught Expected Pickling Error:\n -> {type(e).__name__}: {e}")
finally:
    logger.file_handle.close()
    if os.path.exists("/tmp/experiment_alpha.log"):
        os.remove("/tmp/experiment_alpha.log")

### 2. Custom State Control: `__getstate__` and `__setstate__`
To cleanly handle transient resources:
- **`__getstate__(self)`:** Returns a filtered dictionary of state to be saved. We omit open files, sockets, locks, or bulky caches.
- **`__setstate__(self, state)`:** Accepts the restored dictionary, updates `self.__dict__`, and re-creates fresh runtime resources (e.g. reopens log files, reinitializes thread locks).

In [ ]:
import __main__

class SafeModelLogger:
    # Production-ready stateful logger with pickle hooks.
    def __init__(self, log_name):
        self.log_name = log_name
        self.log_path = f"/tmp/{log_name}.log"
        self._init_runtime_resources()

    def _init_runtime_resources(self):
        # Re-initialize transient runtime handle
        self.file_handle = open(self.log_path, "a")

    def log(self, message):
        self.file_handle.write(f"[{time.strftime('%H:%M:%S')}] {message}\n")
        self.file_handle.flush()

    def __getstate__(self):
        # Copy dictionary and remove unpicklable file handle
        state = self.__dict__.copy()
        del state["file_handle"]
        return state

    def __setstate__(self, state):
        # Restore serialized attributes
        self.__dict__.update(state)
        # Re-establish transient file handle upon deserialization
        self._init_runtime_resources()

__main__.SafeModelLogger = SafeModelLogger

# Test Safe Pickling
safe_logger = SafeModelLogger("experiment_beta")
safe_logger.log("Phase 1 complete.")

# Pickling now succeeds seamlessly
serialized_logger = pickle.dumps(safe_logger)
print(f"Successfully serialized SafeModelLogger ({len(serialized_logger)} bytes)!")

# Unpickle in another context
restored_logger = pickle.loads(serialized_logger)
restored_logger.log("Phase 2 resumed from checkpoint.")
print("Successfully logged to restored file handle!")

# Clean up
safe_logger.file_handle.close()
restored_logger.file_handle.close()
if os.path.exists(safe_logger.log_path):
    os.remove(safe_logger.log_path)

### 3. Class Schema Evolution & Versioning
When deployed machine learning models are saved to disk, software code changes over time (e.g. renaming an attribute `weights` $\to$ `coefficients`, adding a new configuration flag).
Using `__setstate__` allows you to inspect incoming state dictionaries, detect schema version mismatches, and apply automated migration transforms on-the-fly.

In [ ]:
class ResilientClassifier:
    # Machine learning model class supporting schema version migrations.
    CURRENT_VERSION = 2

    def __init__(self, weights, regularization=0.01):
        self.version = self.CURRENT_VERSION
        self.weights = np.array(weights, dtype=np.float32)
        self.regularization = regularization

    def __setstate__(self, state):
        saved_version = state.get("version", 1)

        # Migrate Schema Version 1 -> Version 2:
        # Version 1 stored 'w' instead of 'weights' and lacked 'regularization'
        if saved_version == 1:
            print(" -> [MIGRATION] Detected legacy Version 1 state. Migrating attributes...")
            state["weights"] = np.array(state.pop("w"), dtype=np.float32)
            state["regularization"] = 0.01 # Supply default for new attribute
            state["version"] = 2

        self.__dict__.update(state)

__main__.ResilientClassifier = ResilientClassifier

# Simulate legacy Version 1 state dictionary created by an older codebase
legacy_state = {
    "version": 1,
    "w": [0.45, -1.2, 0.88]
}
legacy_bytes = pickle.dumps(legacy_state)

# Wrap inside a simulated pickle stream
# When deserializing with modern class definition:
restored_model = ResilientClassifier([1.0, 2.0])
restored_model.__setstate__(legacy_state)

print(f"Migrated Model Version: {restored_model.version}")
print(f"Migrated Weights:       {restored_model.weights}")
print(f"Injected Default Reg:   {restored_model.regularization}")

### 4. Advanced: The `__reduce__` Protocol
The **`__reduce__()`** method is the lowest-level serialization protocol in Python.
Instead of returning a state dictionary, `__reduce__()` returns a tuple:
$$(\text{callable\_factory}, (\text{arg}_1, \text{arg}_2, \dots), \text{optional\_state})$$
- When unpickling, Python executes `callable_factory(*args)` to instantiate the object.
- This allows you to reconstruct objects through specific factory functions, singletons, or enforce custom validation guards.

In [ ]:
def rebuild_bounded_array(shape, min_val, max_val, data):
    # Factory reconstruction function
    arr = np.array(data).reshape(shape)
    # Enforce invariant bounds check during reconstruction
    clipped = np.clip(arr, min_val, max_val)
    return clipped

class BoundedMatrix:
    def __init__(self, matrix, min_val=-5.0, max_val=5.0):
        self.min_val = min_val
        self.max_val = max_val
        self.data = np.clip(matrix, min_val, max_val)

    def __reduce__(self):
        # Specify exact factory callable and arguments
        factory = rebuild_bounded_array
        args = (self.data.shape, self.min_val, self.max_val, self.data.flatten().tolist())
        return (factory, args)

__main__.rebuild_bounded_array = rebuild_bounded_array
__main__.BoundedMatrix = BoundedMatrix

# Instantiate and test reduce pickling
bm = BoundedMatrix(np.array([[10.0, -12.0], [2.5, 3.1]]), min_val=-5.0, max_val=5.0)
bm_pickled = pickle.dumps(bm)

restored_arr = pickle.loads(bm_pickled)
print("Reconstructed array via __reduce__ factory:")
print(restored_arr)